# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 8 — Evaluación y mejora de RAG

ITBA Educación Ejecutiva

Medir antes de optimizar: hasta ahora diagnosticamos caso por caso (Clase 7) — hoy construimos un dataset de evaluación y métricas, para saber objetivamente si una mejora realmente mejora algo.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo. No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 11 (Laboratorio)**: checkpoint del proyecto integrador.

## Objetivo de la clase

Al final de la clase vas a poder:
- Construir un dataset de evaluación con evidencia esperada por pregunta.
- Calcular métricas de recuperación (hit rate, recall, precision@k, MRR) y de generación (fidelidad, relevancia) con LLM-as-a-Judge.
- Medir costo y latencia agregados, y clasificar errores por etapa.
- Aplicar búsqueda híbrida (BM25 + semántica) y reranking, y demostrar con números si mejoran el sistema.

**Checkpoint del proyecto integrador de hoy**: dataset de evaluación armado y métricas base (fidelidad, relevancia, costo, latencia) medidas (ver `proyecto-integrador/README.md`, checkpoint Clase 8).

## Agenda

1. Medir antes de optimizar y construcción del dataset de evaluación
2. Métricas de recuperación: hit rate, recall, precision@k, MRR
3. Métricas de generación: fidelidad y relevancia (LLM-as-a-Judge)
4. Evaluación humana vs. automática
5. Costo y latencia agregados
6. Análisis de errores por etapa
7. RAGAS (contenido complementario)
8. Mejora: búsqueda híbrida (BM25 + semántica)
9. Mejora: reranking
10. Aplicar una mejora e informe comparativo
11. Laboratorio: medición del sistema base y comparación

## 0. Punto de partida

Reconstruimos el pipeline RAG de las Clases 6-7 (mismo corpus de juguete, con el chunk redundante `apis-003`) para tener algo completo sobre qué medir.

In [1]:
import os, json, time
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos, input_type="search_document"):
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    return json.loads(resp["body"].read())["embeddings"]["float"]

def llamar_bedrock(prompt, system=None, temperature=0.0, max_tokens=500):
    kwargs = {"modelId": MODEL_ID, "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens}}
    if system:
        kwargs["system"] = [{"text": system}]
    resp = bedrock.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"], resp["usage"], resp["metrics"]["latencyMs"]

In [2]:
import chromadb

corpus = [
    {"id": "apis-001", "texto": "La Converse API de Bedrock unifica la forma de llamar a distintos modelos: cambia el modelId, no el código.", "doc": "guia-apis"},
    {"id": "apis-002", "texto": "Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.", "doc": "guia-apis"},
    {"id": "apis-003", "texto": "La temperatura y el máximo de tokens se configuran dentro del campo inferenceConfig de Bedrock.", "doc": "guia-apis"},
    {"id": "seguridad-001", "texto": "La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.", "doc": "guia-seguridad"},
    {"id": "seguridad-002", "texto": "El principio de mínimo privilegio limita qué herramientas puede invocar un agente según el contexto.", "doc": "guia-seguridad"},
    {"id": "rag-001", "texto": "Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.", "doc": "guia-rag"},
    {"id": "rag-002", "texto": "El top-k determina cuántos fragmentos recuperados se incluyen como contexto para el modelo generador.", "doc": "guia-rag"},
]

cliente_chroma = chromadb.Client()
coleccion = cliente_chroma.get_or_create_collection("corpus_clase8")
coleccion.add(
    ids=[c["id"] for c in corpus],
    embeddings=embed_bedrock([c["texto"] for c in corpus]),
    documents=[c["texto"] for c in corpus],
    metadatas=[{"doc": c["doc"]} for c in corpus],
)

def recuperar(consulta, k=3):
    vec = embed_bedrock([consulta], input_type="search_query")[0]
    r = coleccion.query(query_embeddings=[vec], n_results=k)
    return [{"chunk_id": i, "texto": t, "doc": m["doc"], "distancia": d}
            for i, t, m, d in zip(r["ids"][0], r["documents"][0], r["metadatas"][0], r["distances"][0])]

def rag(consulta, k=3, umbral=1.0):
    chunks = [c for c in recuperar(consulta, k) if c["distancia"] < umbral]
    if not chunks:
        return {"respuesta": "No encontré evidencia suficiente para responder.", "chunks": [], "usage": None, "latencia_ms": 0}
    contexto = "\n\n".join(f"[{i+1}] {c['texto']}" for i, c in enumerate(chunks))
    prompt = f"<evidencia>\n{contexto}\n</evidencia>\n\n<pregunta>\n{consulta}\n</pregunta>\n\nRespondé usando solo la evidencia."
    system = "Respondé EXCLUSIVAMENTE con la evidencia provista. Si no alcanza, decilo explícitamente."
    respuesta, usage, latencia = llamar_bedrock(prompt, system=system)
    return {"respuesta": respuesta, "chunks": chunks, "usage": usage, "latencia_ms": latencia}

print("Pipeline reconstruido. Corpus:", coleccion.count(), "chunks")

Pipeline reconstruido. Corpus: 7 chunks


## 1. Medir antes de optimizar y dataset de evaluación

Sin un dataset de evaluación, "mejorar" el RAG es adivinar — cambiás algo, te parece que anduvo mejor en 2-3 pruebas sueltas, y seguís. Un dataset fijo (preguntas + evidencia esperada) permite comparar versiones de forma objetiva y repetible.

Cada caso necesita, como mínimo: la pregunta, qué chunk(s) deberían recuperarse, y si se espera que el sistema responda o diga "no sé".

In [3]:
dataset_evaluacion = [
    {"pregunta": "¿Qué parámetro controla la temperatura de generación?", "chunks_relevantes": ["apis-002", "apis-003"], "debe_responder": True},
    {"pregunta": "¿Qué es la inyección de prompts?", "chunks_relevantes": ["seguridad-001"], "debe_responder": True},
    {"pregunta": "¿Qué elementos describen un pipeline RAG?", "chunks_relevantes": ["rag-001", "rag-002"], "debe_responder": True},
    {"pregunta": "¿Qué es el principio de mínimo privilegio?", "chunks_relevantes": ["seguridad-002"], "debe_responder": True},
    {"pregunta": "¿Cómo se hace un backup de la base de datos?", "chunks_relevantes": [], "debe_responder": False},
]
print(f"{len(dataset_evaluacion)} casos en el dataset de evaluación.")

5 casos en el dataset de evaluación.


**🏠 Practicá**: agregá 3 casos más sobre tu corpus real del TP (MDN Web Docs), incluyendo uno con más de un chunk relevante — lo vas a necesitar para el laboratorio.

## 2. Métricas de recuperación

- **Hit rate@k**: ¿al menos un chunk relevante está en el top-k? (sí/no por pregunta, promediado)
- **Recall@k**: de los chunks relevantes que existen, ¿qué fracción se recuperó?
- **Precision@k**: de los k chunks recuperados, ¿qué fracción es relevante?
- **MRR** (Mean Reciprocal Rank): en qué posición apareció el primer chunk relevante (1/posición, promediado) — premia que lo relevante esté *arriba*, no solo presente.

In [4]:
def metricas_recuperacion(dataset, k=3):
    hits, recalls, precisions, reciprocal_ranks = [], [], [], []

    for caso in dataset:
        if not caso["chunks_relevantes"]:
            continue  # estos casos se evalúan aparte (sección 4: ¿dijo "no sé" correctamente?)

        recuperados = [c["chunk_id"] for c in recuperar(caso["pregunta"], k=k)]
        relevantes = set(caso["chunks_relevantes"])
        recuperados_relevantes = [r for r in recuperados if r in relevantes]

        hits.append(1 if recuperados_relevantes else 0)
        recalls.append(len(set(recuperados_relevantes)) / len(relevantes))
        precisions.append(len(recuperados_relevantes) / k)

        rr = 0.0
        for pos, r in enumerate(recuperados, start=1):
            if r in relevantes:
                rr = 1 / pos
                break
        reciprocal_ranks.append(rr)

    return {
        "hit_rate": sum(hits) / len(hits),
        "recall": sum(recalls) / len(recalls),
        "precision": sum(precisions) / len(precisions),
        "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks),
    }


metricas_base = metricas_recuperacion(dataset_evaluacion, k=3)
for nombre, valor in metricas_base.items():
    print(f"{nombre}: {valor:.3f}")

hit_rate: 1.000
recall: 1.000
precision: 0.500
mrr: 1.000


Notá la diferencia entre recall y precision en el caso "¿Qué elementos describen un pipeline RAG?" (2 chunks relevantes, `rag-001` y `rag-002`): si el top-3 trae ambos más un chunk irrelevante, recall=100% pero precision=66%. Son medidas de cosas distintas — alto recall con baja precision significa "no te perdiste nada, pero metiste ruido".

**🏠 Practicá**: corré `metricas_recuperacion` con k=1 y k=5, y compará. ¿Qué métrica sube y cuál baja al aumentar k? ¿Tiene sentido con lo que viste en la Clase 7 sobre ajuste de top-k?

## 3. Métricas de generación: fidelidad y relevancia (LLM-as-a-Judge)

Las métricas de recuperación miden si *llegó* la evidencia correcta — no si la *respuesta* la usó bien. Para eso hace falta evaluar el texto generado, y la forma más práctica de automatizarlo es usar otro LLM como juez: **LLM-as-a-Judge**.

- **Fidelidad (faithfulness)**: ¿la respuesta está sustentada en la evidencia, sin agregar nada que no esté ahí?
- **Relevancia (answer relevance)**: ¿la respuesta realmente contesta la pregunta (más allá de estar sustentada)?

In [23]:
from pydantic import BaseModel, Field
from typing import Literal

class JuicioLLM(BaseModel):
    fidelidad: Literal[1, 2, 3, 4, 5] = Field(..., description="1=inventa información no presente en la evidencia, 5=totalmente sustentada")
    relevancia: Literal[1, 2, 3, 4, 5] = Field(..., description="1=no contesta la pregunta, 5=la contesta completamente")
    justificacion: str = Field(..., description="Una oración explicando el puntaje")


SYSTEM_JUEZ = (
    "Sos un evaluador estricto de sistemas RAG. Vas a recibir una pregunta, la evidencia que tenía disponible "
    "el sistema, y la respuesta que generó. Evaluá fidelidad y relevancia según la evidencia, no según tu "
    "propio conocimiento. Respondé ÚNICAMENTE con un JSON que cumpla el schema provisto."
    "Para la justificación utilizá solo texto plano sin ningún símbolo salvo los signos de puntuación."
)

def juzgar(pregunta: str, evidencia: str, respuesta: str) -> JuicioLLM:
    prompt = (
        f"Pregunta: {pregunta}\n\nEvidencia disponible:\n{evidencia}\n\nRespuesta generada:\n{respuesta}\n\n"
        f"Schema: {JuicioLLM.model_json_schema()}"
    )
    texto, _, _ = llamar_bedrock(prompt, system=SYSTEM_JUEZ, temperature=0.0)
    return JuicioLLM.model_validate_json(texto)


caso = dataset_evaluacion[1]
print(caso)
resultado = rag(caso["pregunta"], umbral=1.0)
print(resultado)
evidencia_texto = "\n".join(c["texto"] for c in resultado["chunks"])
print(evidencia_texto)
juicio = juzgar(caso["pregunta"], evidencia_texto, resultado["respuesta"])
print(f"Fidelidad: {juicio.fidelidad}/5  Relevancia: {juicio.relevancia}/5")
print(f"Justificación: {juicio.justificacion}")

{'pregunta': '¿Qué es la inyección de prompts?', 'chunks_relevantes': ['seguridad-001'], 'debe_responder': True}
{'respuesta': 'Según la evidencia proporcionada, la **inyección de prompts** es un ataque que ocurre cuando **contenido no confiable manipula las instrucciones del modelo**.', 'chunks': [{'chunk_id': 'seguridad-001', 'texto': 'La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.', 'doc': 'guia-seguridad', 'distancia': 0.7681365013122559}], 'usage': {'inputTokens': 107, 'outputTokens': 46, 'totalTokens': 153, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}, 'latencia_ms': 1101}
La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.
Fidelidad: 4/5  Relevancia: 5/5
Justificación: La respuesta reproduce fielmente la evidencia disponible y contesta la pregunta directamente, aunque agrega la palabra ataque que no figura explicitamente en la evidencia.


Notá que `juzgar` reutiliza structured outputs + Pydantic de la Clase 3 — LLM-as-a-Judge no es más que otra llamada al modelo con un schema de salida, aplicada a evaluar en vez de a responder.

**🏠 Practicá**: corré `juzgar` sobre los 4 casos del dataset que esperan respuesta y promediá fidelidad/relevancia — es tu primera métrica de generación agregada.

## 4. Evaluación humana vs. automática

LLM-as-a-Judge es rápido y escala, pero la investigación sobre esta técnica documenta varios sesgos sistemáticos, no solo "a veces se equivoca al azar":

- **Sesgo de posición**: cuando se le pide comparar dos respuestas (A vs. B), el juez tiende a favorecer sistemáticamente la que aparece primera (o última) en el prompt, independientemente de cuál es mejor. Mitigación estándar: evaluar en ambos órdenes (A,B) y (B,A), y promediar.
- **Sesgo de verbosidad**: respuestas más largas tienden a puntuarse más alto, incluso cuando no son más correctas ni más útiles — el juez confunde "exhaustivo" con "bueno".
- **Sesgo de auto-preferencia (self-enhancement)**: un modelo tiende a calificar mejor las respuestas generadas por sí mismo o por modelos de la misma familia, que las de otros proveedores — por eso conviene que el juez y el modelo evaluado no sean literalmente el mismo si se puede evitar.
- **Sesgo de formato**: el juez tiende a premiar respuestas con el estilo/formato con el que el propio juez fue entrenado a producir (ej. uso de listas, cierta extensión), más que el contenido en sí.

Ninguno de estos sesgos invalida LLM-as-a-Judge — lo que invalida es usarlo **sin saber que existen**. En la práctica: usarlo para iterar rápido (como en la sección 3), pero validar con una muestra de evaluación humana antes de confiar en una mejora para producción, y ser especialmente cauteloso si el juez y el modelo evaluado comparten proveedor (sesgo de auto-preferencia).

## 5. Costo y latencia agregados

Una mejora que sube la fidelidad 10% pero duplica el costo por consulta puede no valer la pena — siempre se evalúa junto con costo y latencia, no la calidad sola.

In [17]:
PRECIO_POR_MILLON_USD = {"input": 3.0, "output": 15.0}  # valores de ejemplo, verificar precio vigente

def costo_usd(usage):
    if not usage:
        return 0.0
    return usage["inputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["input"] + usage["outputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["output"]


costos, latencias = [], []
for caso in dataset_evaluacion:
    r = rag(caso["pregunta"])
    costos.append(costo_usd(r["usage"]))
    latencias.append(r["latencia_ms"])

print(f"Costo total del dataset: ${sum(costos):.5f}  (promedio: ${sum(costos)/len(costos):.5f} por consulta)")
print(f"Latencia promedio: {sum(latencias)/len(latencias):.0f} ms")

Costo total del dataset: $0.00344  (promedio: $0.00069 por consulta)
Latencia promedio: 700 ms


## 6. Análisis de errores por etapa

Reutilizamos el framework de la Clase 7: para cada caso del dataset con métricas bajas, clasificar en qué etapa está el problema antes de intentar arreglar nada.

In [26]:
def clasificar_error(caso, k=3):
    if not caso["chunks_relevantes"]:
        return None  # casos "no debe responder" se evalúan distinto

    recuperados = [c["chunk_id"] for c in recuperar(caso["pregunta"], k=k)]
    if not any(r in caso["chunks_relevantes"] for r in recuperados):
        return "RECUPERACIÓN: ningún chunk relevante llegó al top-k"

    resultado = rag(caso["pregunta"], k=k, umbral=0.8)
    chunks_en_contexto = [c["chunk_id"] for c in resultado["chunks"]]
    if not any(r in chunks_en_contexto for r in caso["chunks_relevantes"]):
        return "CONTEXTO: el chunk relevante se recuperó pero no pasó el umbral"

    juicio = juzgar(caso["pregunta"], "\n".join(c["texto"] for c in resultado["chunks"]), resultado["respuesta"])
    if juicio.fidelidad < 4:
        return f"GENERACIÓN: fidelidad baja ({juicio.fidelidad}/5) pese a tener la evidencia correcta"

    return None  # sin error detectado


for caso in dataset_evaluacion:
    diagnostico = clasificar_error(caso)
    if diagnostico:
        print(f"⚠ {caso['pregunta']}\n  -> {diagnostico}")
    else:
        print(f"✓ {caso['pregunta']}")

⚠ ¿Qué parámetro controla la temperatura de generación?
  -> CONTEXTO: el chunk relevante se recuperó pero no pasó el umbral
✓ ¿Qué es la inyección de prompts?
⚠ ¿Qué elementos describen un pipeline RAG?
  -> CONTEXTO: el chunk relevante se recuperó pero no pasó el umbral
⚠ ¿Qué es el principio de mínimo privilegio?
  -> CONTEXTO: el chunk relevante se recuperó pero no pasó el umbral
✓ ¿Cómo se hace un backup de la base de datos?


## 7. RAGAS

Lo que armamos a mano en las secciones 2-3 son, esencialmente, versiones simplificadas de métricas que **RAGAS** (librería open source especializada en evaluación de RAG) ya implementa de forma estandarizada:

| Lo que armamos a mano (secciones 2-3) | Métrica equivalente en RAGAS | Qué mide |
|---|---|---|
| `metricas_recuperacion` → `recall` | `context_recall` | De los chunks relevantes que existen, ¿cuántos se recuperaron? |
| (no lo armamos, RAGAS sí lo tiene) | `context_precision` | De los chunks recuperados, ¿cuántos eran realmente relevantes y bien rankeados? |
| `juzgar().fidelidad` | `faithfulness` | ¿La respuesta está sustentada en el contexto, sin agregar nada? |
| `juzgar().relevancia` | `answer_relevancy` | ¿La respuesta realmente contesta la pregunta? |

La API interna de RAGAS cambia seguido entre versiones (conviven una API "legacy" basada en `SingleTurnSample`/`ragas.metrics` y una nueva basada en `ragas.metrics.collections`) — lo que viene abajo usa el camino **oficialmente documentado para Amazon Bedrock** (vía `langchain-aws`), que es estable incluso cuando otras partes de la librería cambien.

In [28]:
# %pip install ragas langchain-aws

from langchain_aws import ChatBedrockConverse
from ragas.llms import LangchainLLMWrapper

# RAGAS necesita un wrapper de LangChain alrededor del modelo -- no llama a boto3 directamente.
bedrock_para_ragas = ChatBedrockConverse(model=MODEL_ID, region_name=os.environ.get("AWS_REGION", "us-east-1"))
evaluator_llm = LangchainLLMWrapper(bedrock_para_ragas)
print("evaluator_llm listo.")

evaluator_llm listo.


/tmp/ipykernel_72141/945674043.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(bedrock_para_ragas)


### Faithfulness con RAGAS, sobre un caso real

Comparamos la métrica de RAGAS contra nuestro propio `juzgar().fidelidad` (sección 3) para el mismo caso — si miden cosas parecidas, deberían dar resultados compatibles.

In [29]:
from ragas.metrics import Faithfulness
from ragas.dataset_schema import SingleTurnSample

caso = dataset_evaluacion[0]  # "¿Qué parámetro controla la temperatura de generación?"
resultado_rag = rag(caso["pregunta"])

sample = SingleTurnSample(
    user_input=caso["pregunta"],
    response=resultado_rag["respuesta"],
    retrieved_contexts=[c["texto"] for c in resultado_rag["chunks"]],
)

faithfulness_scorer = Faithfulness(llm=evaluator_llm)
score_ragas = await faithfulness_scorer.single_turn_ascore(sample)

juicio_propio = juzgar(caso["pregunta"], "\n".join(c["texto"] for c in resultado_rag["chunks"]), resultado_rag["respuesta"])

print(f"Faithfulness (RAGAS, escala 0-1):    {score_ragas:.3f}")
print(f"Fidelidad (nuestro juez, escala 1-5): {juicio_propio.fidelidad}/5")

/tmp/ipykernel_72141/3780069126.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


Faithfulness (RAGAS, escala 0-1):    1.000
Fidelidad (nuestro juez, escala 1-5): 5/5


Notá las dos diferencias de diseño entre ambas: RAGAS devuelve un score continuo 0-1 (internamente, descompone la respuesta en afirmaciones atómicas y verifica cuántas están sustentadas en el contexto — más granular que pedirle al juez un número del 1 al 5 de una sola vez), y usa su propio prompt interno de evaluación (no el que escribimos en `SYSTEM_JUEZ`). Que ambas coincidan en la dirección general (alta fidelidad en un caso bien sustentado) es una buena señal de que nuestra versión a mano no estaba midiendo algo completamente distinto — pero no van a dar el mismo número exacto, y no tiene por qué pasar.

### Un criterio propio con `AspectCritic`

RAGAS también permite definir **criterios de evaluación propios** en lenguaje natural, sin tener que programar la lógica de scoring — útil cuando ninguna métrica estándar captura exactamente lo que te importa en tu dominio.

In [30]:
from ragas.metrics import AspectCritic

cita_valida = AspectCritic(
    name="cita_valida",
    llm=evaluator_llm,
    definition=(
        "Devolvé 1 si la respuesta incluye al menos una cita entre corchetes (ej. [1]) que efectivamente "
        "corresponde a información presente en el contexto. Devolvé 0 si no cita ninguna fuente, o si cita "
        "algo que el contexto no respalda."
    ),
)

score_citas = await cita_valida.single_turn_ascore(sample)
print(f"¿Cita de forma válida? (1=sí, 0=no): {score_citas}")

/tmp/ipykernel_72141/3351777276.py:1: DeprecationWarning: Importing AspectCritic from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AspectCritic
  from ragas.metrics import AspectCritic


¿Cita de forma válida? (1=sí, 0=no): 0


**🏠 Practicá**: corré `Faithfulness` sobre los otros casos de `dataset_evaluacion` que sí esperan respuesta (no solo el primero) y promediá los scores — es la versión con RAGAS de lo que ya hiciste a mano en la sección 3. Si tu proveedor del TP no es Bedrock, `LangchainLLMWrapper` también envuelve `ChatOpenAI`/`ChatAnthropic` de `langchain-openai`/`langchain-anthropic` — mismo patrón, otro import.

## 8. Mejora: búsqueda híbrida (BM25 + semántica)

La búsqueda semántica (embeddings) es buena para significado, pero puede fallar con términos exactos poco frecuentes (nombres de funciones, códigos de error) que casi no tienen "significado" distintivo en el espacio vectorial. **BM25** es un algoritmo clásico de búsqueda por palabras clave (basado en frecuencia de términos) que es complementario, no competidor.

**Búsqueda híbrida**: combinar el ranking de ambos métodos (por ejemplo, con *Reciprocal Rank Fusion*) en vez de elegir uno solo.

### La fórmula de BM25

BM25 pondera cada término de la consulta según cuánto aparece en el documento, pero con dos correcciones sobre un conteo ingenuo de frecuencia:

$$
\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$

- $f(q_i, D)$: cuántas veces aparece el término $q_i$ de la consulta en el documento $D$.
- $\text{IDF}(q_i)$ (*inverse document frequency*): términos que aparecen en pocos documentos del corpus pesan más que términos comunes a casi todos — un término raro que matchea es más informativo que uno frecuente.
- $k_1$ (típicamente 1.2–2.0): controla la **saturación** de la frecuencia de término — que una palabra aparezca 10 veces en vez de 1 no multiplica el score por 10; el aporte marginal de cada repetición adicional decae.
- $b$ (típicamente 0.75): controla cuánto se penaliza a los documentos más largos que el promedio del corpus (`avgdl`) — sin esto, un documento simplemente más largo acumularía más matches de términos por azar, no por ser más relevante.

No hace falta memorizar la fórmula para usar `BM25Okapi` — pero entender qué hacen $k_1$ y $b$ ayuda a diagnosticar resultados raros (ej. si un documento larguísimo siempre gana, puede ser necesario ajustar `b`).

In [34]:
# %pip install rank_bm25
from rank_bm25 import BM25Okapi

corpus_tokenizado = [c["texto"].lower().split() for c in corpus]
bm25 = BM25Okapi(corpus_tokenizado)

def buscar_bm25(consulta: str, k: int = 3) -> list[dict]:
    scores = bm25.get_scores(consulta.lower().split())
    indices_top = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]
    return [{"chunk_id": corpus[i]["id"], "score_bm25": scores[i]} for i in indices_top]


def busqueda_hibrida(consulta: str, k: int = 3, k_fusion: int = 60) -> list[str]:
    """Reciprocal Rank Fusion: combina el ranking semántico y el de BM25."""
    ranking_semantico = [c["chunk_id"] for c in recuperar(consulta, k=len(corpus))]
    ranking_bm25 = [r["chunk_id"] for r in buscar_bm25(consulta, k=len(corpus))]

    scores_fusion = {}
    for ranking in (ranking_semantico, ranking_bm25):
        for pos, chunk_id in enumerate(ranking, start=1):
            scores_fusion[chunk_id] = scores_fusion.get(chunk_id, 0) + 1 / (k_fusion + pos)

    return sorted(scores_fusion, key=lambda cid: -scores_fusion[cid])[:k]


print("Híbrido:", busqueda_hibrida("¿cómo configuro el inferenceConfig?", k=3))
print("Solo semántico:", [c["chunk_id"] for c in recuperar("¿cómo configuro el inferenceConfig?", k=3)])

Híbrido: ['rag-002', 'apis-002', 'apis-003']
Solo semántico: ['apis-002', 'apis-003', 'seguridad-001']


### Por qué Reciprocal Rank Fusion, y qué hace la constante $k$

RRF combina dos rankings (semántico y BM25) sin necesitar normalizar sus scores a la misma escala — algo que de otra forma sería un problema, porque una distancia coseno y un score de BM25 no son comparables directamente. En cambio, RRF usa solo la **posición** de cada documento en cada ranking:

$$
\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}
$$

La constante $k$ (comúnmente 60, el valor usado en el paper original que propuso el método) suaviza la influencia de las posiciones más altas: sin ella, un documento que queda primero en un solo ranking dominaría el resultado final aunque no aparezca en el otro ranking en absoluto. Con $k$ grande, importa más aparecer **razonablemente bien ubicado en varios rankings** que ser el número 1 en uno solo — que es, justamente, el comportamiento que queremos de una fusión.

**🏠 Practicá**: probá una consulta con un término técnico muy específico y poco frecuente (ej. el nombre exacto de un parámetro, "maxTokens") y compará búsqueda híbrida vs. solo semántica — es el caso donde BM25 suele ganar, porque coincide la palabra exacta.

## 9. Mejora: reranking

Ya vimos el concepto en la Clase 1 (bi-encoder vs. cross-encoder). AWS Bedrock expone un **modelo de reranking real** (Cohere Rerank 3.5) vía una API dedicada — se usa como segunda etapa, sobre los candidatos que ya trajo la recuperación inicial (semántica o híbrida).

In [33]:
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))

def rerank(consulta: str, candidatos: list[dict], top_n: int = 3) -> list[dict]:
    response = bedrock_agent_runtime.rerank(
        queries=[{"type": "TEXT", "textQuery": {"text": consulta}}],
        sources=[
            {"type": "INLINE", "inlineDocumentSource": {"type": "TEXT", "textDocument": {"text": c["texto"]}}}
            for c in candidatos
        ],
        rerankingConfiguration={
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "modelConfiguration": {"modelArn": "arn:aws:bedrock:us-east-1::foundation-model/cohere.rerank-v3-5:0"},
                "numberOfResults": top_n,
            },
        },
    )
    return [{"chunk_id": candidatos[r["index"]]["chunk_id"], "relevance_score": r["relevanceScore"]} for r in response["results"]]


candidatos = recuperar("¿qué controla la temperatura de generación?", k=6)  # traemos más candidatos de los que necesitamos
print(candidatos)
reordenados = rerank("¿qué es mínimo privilegio?", candidatos, top_n=3)
print(reordenados)
for r in reordenados:
    print(f"{r['chunk_id']}  (score={r['relevance_score']:.3f})")

[{'chunk_id': 'apis-003', 'texto': 'La temperatura y el máximo de tokens se configuran dentro del campo inferenceConfig de Bedrock.', 'doc': 'guia-apis', 'distancia': 1.4733667373657227}, {'chunk_id': 'apis-002', 'texto': 'Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.', 'doc': 'guia-apis', 'distancia': 1.5474246740341187}, {'chunk_id': 'rag-002', 'texto': 'El top-k determina cuántos fragmentos recuperados se incluyen como contexto para el modelo generador.', 'doc': 'guia-rag', 'distancia': 1.6234837770462036}, {'chunk_id': 'rag-001', 'texto': 'Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.', 'doc': 'guia-rag', 'distancia': 1.67086660861969}, {'chunk_id': 'seguridad-001', 'texto': 'La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.', 'doc': 'guia-seguridad', 'distancia': 1.7836461067199707}, {'chunk_id': 'seguridad-002', 'texto': 'El principio de

Notá el patrón: recuperación inicial trae k_grande candidatos "razonablemente relevantes" rápido (bi-encoder, barato), reranking reordena con un modelo más preciso pero más lento (cross-encoder) — exactamente el trade-off de la Clase 1. No conviene rerankear todo el corpus, solo el top de la recuperación inicial.

**🏠 Practicá**: compará el orden de `reordenados` contra el orden semántico simple de `candidatos` — ¿cambió algo? Con nuestro corpus de juguete (pocos candidatos, bien separados) puede no notarse mucho; con el corpus real del TP debería ser más evidente.

## 10. Aplicar una mejora e informe comparativo

El punto de todo lo anterior: no alcanza con "creo que el reranking ayuda" — hay que medirlo contra el mismo dataset de evaluación, antes y después.

In [ ]:
def metricas_con_reranking(dataset, k_candidatos=6, k_final=3):
    hits = []
    for caso in dataset:
        if not caso["chunks_relevantes"]:
            continue
        candidatos = recuperar(caso["pregunta"], k=k_candidatos)
        reordenados = rerank(caso["pregunta"], candidatos, top_n=k_final)
        recuperados = [r["chunk_id"] for r in reordenados]
        hits.append(1 if any(r in caso["chunks_relevantes"] for r in recuperados) else 0)
    return {"hit_rate": sum(hits) / len(hits)}


metricas_mejoradas = metricas_con_reranking(dataset_evaluacion)

print("--- Informe comparativo ---")
print(f"Hit rate ANTES (solo semántico, k=3):     {metricas_base['hit_rate']:.3f}")
print(f"Hit rate DESPUÉS (con reranking, k=3):     {metricas_mejoradas['hit_rate']:.3f}")

Con un corpus de juguete de 7 chunks, es esperable que no haya diferencia (ya partíamos de hit rate alto) — el valor real de este informe aparece con el corpus real del TP, donde hay más margen para que una mejora se note en los números. El punto pedagógico es el **proceso**: medir antes, aplicar, medir después, comparar — no la magnitud del número en este ejemplo chico.

### ¿La mejora es real o es ruido?

Con un dataset de evaluación chico (5-10 casos, como el nuestro), una diferencia de "0.80 vs. 0.85 de hit rate" puede no significar nada — podría desaparecer, o invertirse, con solo agregar o sacar un caso del dataset. Antes de declarar "la mejora funcionó", conviene preguntarse qué tan sólido es ese número.

Una forma simple de estimarlo sin matemática avanzada: **bootstrap** — remuestrear el dataset (con reposición) muchas veces, recalcular la métrica en cada remuestreo, y mirar el rango donde cae la mayoría de los resultados (intervalo de confianza).

In [ ]:
import random

def bootstrap_ci(valores_binarios: list[int], n_iteraciones: int = 1000, confianza: float = 0.95) -> tuple[float, float]:
    """valores_binarios: 1 si el caso tuvo hit, 0 si no. Devuelve (límite inferior, límite superior) del intervalo."""
    medias_remuestreadas = []
    n = len(valores_binarios)
    for _ in range(n_iteraciones):
        muestra = [random.choice(valores_binarios) for _ in range(n)]
        medias_remuestreadas.append(sum(muestra) / n)

    medias_remuestreadas.sort()
    cola = (1 - confianza) / 2
    lo = medias_remuestreadas[int(cola * n_iteraciones)]
    hi = medias_remuestreadas[int((1 - cola) * n_iteraciones) - 1]
    return lo, hi


def hits_binarios(dataset, k=3) -> list[int]:
    resultado = []
    for caso in dataset:
        if not caso["chunks_relevantes"]:
            continue
        recuperados = [c["chunk_id"] for c in recuperar(caso["pregunta"], k=k)]
        resultado.append(1 if any(r in caso["chunks_relevantes"] for r in recuperados) else 0)
    return resultado


hits_antes = hits_binarios(dataset_evaluacion)
lo, hi = bootstrap_ci(hits_antes)
print(f"Hit rate: {sum(hits_antes)/len(hits_antes):.3f}  |  intervalo de confianza 95%: [{lo:.3f}, {hi:.3f}]")
print(f"(calculado sobre {len(hits_antes)} casos — con tan pocos casos, el intervalo va a ser ancho a propósito)")

Si volvés a correr esto con la versión "mejorada" del sistema (reranking o búsqueda híbrida) y los dos intervalos se **superponen** mucho, la diferencia que viste no alcanza para decir "la mejora funcionó" — necesitarías un dataset más grande para distinguir una mejora real de variación aleatoria. Si los intervalos **no se superponen**, es una señal mucho más sólida.

**🏠 Practicá**: calculá `bootstrap_ci` para la versión con reranking (sección 9) y compará el intervalo contra el de arriba. Con el dataset de juguete de hoy (5 casos) es muy probable que se superpongan casi del todo — es la demostración de por qué el laboratorio pide un dataset real de al menos 10 preguntas: por debajo de eso, cualquier conclusión sobre una mejora es poco confiable.

## 11. Laboratorio: medición del sistema base y comparación

**🏠 Tarea de la semana — entregar/completar antes de la Clase 9.**

**Checkpoint del proyecto integrador (Clase 8)**: dataset de evaluación armado y métricas base (fidelidad, relevancia, costo, latencia) medidas sobre tu RAG real.

### Consigna

1. Armá un dataset de evaluación de al menos 10 preguntas sobre tu corpus real (MDN Web Docs), con `chunks_relevantes` esperados — incluí casos con un solo chunk relevante, casos con varios, y al menos 2 fuera de dominio.
2. Corré `metricas_recuperacion` (hit rate, recall, precision, MRR) sobre tu sistema actual (el de la Clase 7).
3. Corré LLM-as-a-Judge (fidelidad y relevancia) sobre los casos que sí deberían responder, y calculá el promedio.
4. Medí costo total y latencia promedio del dataset completo.
5. Elegí **una** mejora (búsqueda híbrida o reranking) y volvé a medir — armá un informe comparativo antes/después con los 4 números (hit rate/MRR, fidelidad/relevancia promedio, costo, latencia).
6. Calculá el intervalo de confianza (`bootstrap_ci`, sección 10) del hit rate antes y después, y confirmá si se superponen o no — es lo que determina si podés afirmar que la mejora es real con tu dataset.

### Criterio de éxito

- Tenés un número concreto para cada métrica, no una impresión cualitativa ("me pareció que mejoró").
- El informe comparativo muestra explícitamente si la mejora elegida valió la pena considerando también costo/latencia, no solo calidad.
- El informe dice explícitamente si los intervalos de confianza se superponen o no, y qué implica eso para la confiabilidad de la conclusión.
- Podés señalar, para al menos un caso del dataset, en qué etapa está el error (sección 6) si todavía falla después de la mejora.

In [ ]:
def evaluar_sistema(dataset, pipeline_rag) -> dict:
    # TODO: calcular métricas de recuperación (metricas_recuperacion, adaptada a tu pipeline real)
    # TODO: calcular fidelidad/relevancia promedio con LLM-as-a-Judge sobre los casos con respuesta esperada
    # TODO: calcular costo total y latencia promedio
    # TODO: devolver un dict con las 4 categorías de métricas
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# metricas_antes = evaluar_sistema(tu_dataset, tu_rag_sin_mejora)
# metricas_despues = evaluar_sistema(tu_dataset, tu_rag_con_mejora)
# print(metricas_antes, metricas_despues)

## Cierre

- Hoy pasamos de "diagnostiqué caso por caso" (Clase 7) a medir con métricas sobre un dataset completo — recuperación, generación, costo y latencia.
- Toda mejora de acá en adelante (en el TP, y en cualquier sistema RAG real) se evalúa con este mismo patrón: medir antes, aplicar una cosa a la vez, medir después, comparar.
- Próxima clase: herramientas y workflows agénticos — el RAG deja de ser lo único que el sistema sabe hacer.
- Antes de la próxima clase: dejar el laboratorio de la sección 11 funcionando, con el informe comparativo de al menos una mejora.